# 11 · 綜合實作：離開 Notebook

到目前為止所有東西都跑在 notebook 裡。但你不會用 notebook 出貨。

這一章做兩件事：

1. **把 agent 變成一個真正的專案目錄**，用 `adk run` / `adk web` 跑起來
2. **綜合前十一章**，做一個把三層都用上的完整範例

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. ADK 專案長什麼樣子

ADK 的 CLI 有一個硬性約定：

```
my_agent/
├── __init__.py      ← 必須有，而且要 from . import agent
├── agent.py         ← 必須有一個叫 root_agent 的變數
└── .env             ← 金鑰
```

**兩個一定會踩的坑**：

1. `__init__.py` 少了 `from . import agent`，ADK 找不到你的 agent
2. `agent.py` 裡的變數一定要叫 **`root_agent`**，叫別的名字不會被認出來

In [2]:
import shutil
from pathlib import Path

DEMO_DIR = Path.cwd() / "_capstone_demo"
AGENT_DIR = DEMO_DIR / "faq_agent"
shutil.rmtree(DEMO_DIR, ignore_errors=True)
AGENT_DIR.mkdir(parents=True)

(AGENT_DIR / "__init__.py").write_text("from . import agent\n", encoding="utf-8")

(AGENT_DIR / "agent.py").write_text(
    '''"""一個最小但完整的 ADK agent。"""

from google.adk.agents import LlmAgent

_FAQ = {
    "退貨": "商品到貨 7 天內可申請退貨，需保持完整包裝。",
    "運費": "訂單滿 1000 元免運，未滿收 80 元。",
    "發票": "預設開立電子發票，可於訂單頁面改成公司抬頭。",
}


def search_faq(keyword: str) -> dict:
    """依關鍵字查詢常見問題。

    Args:
        keyword: 問題關鍵字，例如 '退貨'、'運費'、'發票'。
    """
    for key, answer in _FAQ.items():
        if key in keyword:
            return {"found": True, "topic": key, "answer": answer}
    return {"found": False, "available": list(_FAQ)}


root_agent = LlmAgent(
    name="faq_agent",
    model="gemini-flash-lite-latest",
    description="回答電商常見問題。",
    instruction=(
        "你是電商客服。使用者提問時一定要先呼叫 search_faq 查詢，"
        "再用繁體中文簡短回答。查不到就告訴使用者你能回答哪些主題。"
    ),
    tools=[search_faq],
)
''',
    encoding="utf-8",
)

# 把金鑰帶進去，讓 CLI 也跑得起來
import os

api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY", "")
(AGENT_DIR / ".env").write_text(f"GOOGLE_API_KEY={api_key}\n", encoding="utf-8")

for path in sorted(DEMO_DIR.rglob("*")):
    print(path.relative_to(DEMO_DIR.parent))

_capstone_demo/faq_agent
_capstone_demo/faq_agent/.env
_capstone_demo/faq_agent/__init__.py
_capstone_demo/faq_agent/agent.py


## 2. 用 `adk run` 跑起來

`adk run` 本來是互動式的，但它讀 stdin，所以可以用管線餵問題進去。
這也是把 agent 接進 CI 的最簡單方式。

In [3]:
import subprocess
import sys

adk_bin = Path(sys.executable).parent / "adk"

result = subprocess.run(
    [str(adk_bin), "run", "faq_agent"],
    cwd=DEMO_DIR,
    input="運費怎麼算？\nexit\n",
    capture_output=True,
    text=True,
    timeout=180,
)

print("--- stdout ---")
print(result.stdout[-1500:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1000:])

--- stdout ---
Log setup complete: /tmp/agents_log/agent.20260904_015042.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
Running agent faq_agent, type exit to exit.
[user]: [faq_agent]: 訂單金額滿 1000 元即享免運優惠，若未滿 1000 元則需收 80 元運費。
[user]: 


### 其他跑法

| 指令 | 用途 |
|---|---|
| `adk run my_agent` | 終端機互動 |
| `adk web` | **開發用 Web UI**，可以看事件串流、state、trace |
| `adk api_server` | 起一個 FastAPI 服務，給前端接 |
| `adk deploy cloud_run \| gke \| agent_engine` | 部署（Day 30） |

`adk web` 是開發階段最有價值的一個——它把我們前面用 `trace=True` 手動印的東西
全部視覺化了。在專案目錄執行：

```bash
adk web
# 然後開 http://localhost:8000
```

> ⚠️ 官方明確標示 **ADK Web 只能用於開發**，不要拿去當正式服務。

## 3. 綜合實作：技術文件產生器

把三層全部用上，做一個「給我一個主題，產出一份技術文件」的系統：

```
  使用者主題
       │
       ▼
 ┌───────────┐
 │ 大綱 agent│ ─ output_key="outline"
 └─────┬─────┘
       ▼
  ┌─────────────────────┐
  │  平行撰寫（三段）    │  ParallelAgent
  │  背景 / 做法 / 陷阱  │
  └──────────┬──────────┘
             ▼
       ┌───────────┐
       │ 組稿 agent│ ─ output_schema=Document
       └───────────┘
```

用到：Layer 1 的 instruction／structured output、Layer 2 的 state、
Layer 3 的 Sequential + Parallel。

In [4]:
from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent
from pydantic import BaseModel, Field

outliner = LlmAgent(
    name="outliner",
    model=get_model(),
    instruction=(
        "你是技術文件規劃師。針對使用者給的主題，列出三個小節標題："
        "一個講背景動機、一個講具體做法、一個講常見陷阱。"
        "只回三行標題，不要編號、不要其他內容。"
    ),
    output_key="outline",
)

背景 = LlmAgent(
    name="background_writer",
    model=get_model(),
    instruction="根據大綱 {outline?}，寫「背景動機」那一節，三句話，繁體中文。",
    output_key="section_background",
)

做法 = LlmAgent(
    name="howto_writer",
    model=get_model(),
    instruction="根據大綱 {outline?}，寫「具體做法」那一節，三個步驟，繁體中文。",
    output_key="section_howto",
)

陷阱 = LlmAgent(
    name="pitfall_writer",
    model=get_model(),
    instruction="根據大綱 {outline?}，寫「常見陷阱」那一節，兩個陷阱，繁體中文。",
    output_key="section_pitfalls",
)


class Document(BaseModel):
    title: str = Field(description="文件標題")
    summary: str = Field(description="一句話摘要")
    sections: list[str] = Field(description="三個小節的完整內容")
    reading_minutes: int = Field(description="預估閱讀分鐘數")


assembler = LlmAgent(
    name="assembler",
    model=get_model(),
    instruction=(
        "把以下三節組成一份文件。\n\n"
        "【背景】\n{section_background?}\n\n"
        "【做法】\n{section_howto?}\n\n"
        "【陷阱】\n{section_pitfalls?}"
    ),
    output_schema=Document,
    output_key="document",
)

doc_generator = SequentialAgent(
    name="doc_generator",
    sub_agents=[
        outliner,
        ParallelAgent(name="writers", sub_agents=[背景, 做法, 陷阱]),
        assembler,
    ],
)


def show_tree(agent, indent=0):
    print("  " * indent + f"{agent.name} ({type(agent).__name__})")
    for child in getattr(agent, "sub_agents", []) or []:
        show_tree(child, indent + 1)


show_tree(doc_generator)

doc_generator (SequentialAgent)
  outliner (LlmAgent)
  writers (ParallelAgent)
    background_writer (LlmAgent)
    howto_writer (LlmAgent)
    pitfall_writer (LlmAgent)
  assembler (LlmAgent)


In [5]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=doc_generator, app_name="capstone")
sid = await new_session(runner)

await ask(runner, "在 Python 專案裡導入型別註記（type hints）", session_id=sid, trace=True)

  💬 [outliner] 為什麼要在 Python 專案中導入型別註記
如何在現有專案中逐步導入型別註記與工具設定
導入型別註記時常見的陷阱與迷思


  💬 [background_writer] 隨著 Python 專案規模日益擴大，動態型別的靈活性逐漸變成維護上的隱患，導致除錯困難與溝通成本上升。導入型別註記能有效提升程式碼的可讀性，並在開發階段提早攔截潛在錯誤。這不僅能強化 IDE 的智慧提示功能，更能為重構大型專案提供堅實的信心與安全網。


  💬 [pitfall_writer] ### 導入型別註記時常見的陷阱與迷思

在 Python 專案中導入型別註記（Type Hints）雖然能大幅提升程式碼的品質與可讀性，但對於習慣動態型別的開發者來說，常常會落入一些常見的陷阱與迷思中。以下是兩個最常遇到的誤區：

#### 陷阱一：追求「百分之百」的型別覆蓋率，導致開發效率停滯
* **迷思說明：** 許多團隊在剛導入型別註記時，會產生一種完美主義心態，認為專案中的每一行程式碼


  💬 [howto_writer] 在 Python 專案中導入型別註記（Type Hints）能有效提升程式碼的品質與可維護性。以下根據大綱，為您詳細說明原因、工具設定步驟，以及常見的陷阱與迷思。

---

### 一、為什麼要在 Python 專案中導入型別註記

Python 雖然是動態型別語言，開發靈活快速，但隨著專案規模擴大，動態型別的缺點也隨之浮現。導入型別註記主要有以下核心好處：

1. **提前發現錯誤（Stati


  💬 [assembler] {
  "title": "在 Python 專案裡導入型別註記",
  "summary": "本文介紹如何在 Python 專案中循序漸進地導入型別註記，提升程式碼品質並避免常見陷阱。",
  "sections": [
    "隨著 Python 專案規模日益擴大，動態型別的靈活性逐漸變成維護上的隱患，導致除錯困難與溝通成本上升。導入型別註記能有效提升程式碼的可讀性，並在開發階段提早攔截潛


'為什麼要在 Python 專案中導入型別註記\n如何在現有專案中逐步導入型別註記與工具設定\n導入型別註記時常見的陷阱與迷思\n隨著 Python 專案規模日益擴大，動態型別的靈活性逐漸變成維護上的隱患，導致除錯困難與溝通成本上升。導入型別註記能有效提升程式碼的可讀性，並在開發階段提早攔截潛在錯誤。這不僅能強化 IDE 的智慧提示功能，更能為重構大型專案提供堅實的信心與安全網。\n### 導入型別註記時常見的陷阱與迷思\n\n在 Python 專案中導入型別註記（Type Hints）雖然能大幅提升程式碼的品質與可讀性，但對於習慣動態型別的開發者來說，常常會落入一些常見的陷阱與迷思中。以下是兩個最常遇到的誤區：\n\n#### 陷阱一：追求「百分之百」的型別覆蓋率，導致開發效率停滯\n* **迷思說明：** 許多團隊在剛導入型別註記時，會產生一種完美主義心態，認為專案中的每一行程式碼、每一個變數和函數都必須加上完整的型別，甚至要求型別檢查工具（如 mypy）不能出現任何警告。\n* **帶來的問題：** Python 本身是一門動態語言，某些進階的動態特性（如高度動態的屬性生成、反射等）要寫出完美的型別註記不僅非常困難，還可能需要寫出冗長且難懂的 `Any` 或複雜的泛型。這會導致開發者花費過多時間在「為了型別而型別」上，反而失去 Python 快速開發的優勢。\n* **正確解法：** 型別註記應該是「漸進式」且「具實用性」的。優先為公開的 API、複雜的商業邏輯函數、以及容易出錯的資料結構加上型別；而對於一些簡單的內部變數或高度動態的程式碼，可以適度放寬標準，不必追求極致的 100% 覆蓋率。\n\n#### 陷阱二：將型別註記等同於「執行期型別安全保證」\n* **迷思說明：** 部分開發者以為在函數參數或回傳值加上了型別（例如 `def add(a: int, b: int) -> int:`），Python 在執行期就會自動檢查並阻擋錯誤型別的輸入。\n* **帶來的問題：** 這是對 Python 型別系統最大的誤解。Python 的型別註記在執行期（Runtime）預設是**完全被忽略**的，它不會進行任何自動轉型或型別檢查。如果傳入錯誤的型別（例如傳入字串 `add("1", "2")`），程式依然可以執行，直到觸發底層運算錯誤為止。\n* **正

### 📌 一個容易搞混的地方

同樣是 `output_schema` 的產物，**拿的位置不同、型別就不同**：

| 來源 | 型別 |
|---|---|
| `run_once()` / `ask()` 的回傳值 | **JSON 字串** → 用 `model_validate_json()` |
| `state[output_key]` | **已經 parse 好的 dict** → 用 `model_validate()` |

對 state 裡的值呼叫 `model_validate_json()` 會得到
`JSON input should be string, bytes or bytearray`。

In [6]:
state = await peek_state(runner, sid)

raw = state["document"]
print("state 裡的型別:", type(raw).__name__)

# state 存的是 dict，所以用 model_validate（不是 model_validate_json）
doc = Document.model_validate(raw) if isinstance(raw, dict) else Document.model_validate_json(raw)

print("=" * 60)
print(f"標題　　: {doc.title}")
print(f"摘要　　: {doc.summary}")
print(f"閱讀時間: {doc.reading_minutes} 分鐘")
for i, section in enumerate(doc.sections, 1):
    print(f"\n--- 第 {i} 節 ---")
    print(section[:300])

state 裡的型別: dict
標題　　: 在 Python 專案裡導入型別註記
摘要　　: 本文介紹如何在 Python 專案中循序漸進地導入型別註記，提升程式碼品質並避免常見陷阱。
閱讀時間: 5 分鐘

--- 第 1 節 ---
隨著 Python 專案規模日益擴大，動態型別的靈活性逐漸變成維護上的隱患，導致除錯困難與溝通成本上升。導入型別註記能有效提升程式碼的可讀性，並在開發階段提早攔截潛在錯誤。這不僅能強化 IDE 的智慧提示功能，更能為重構大型專案提供堅實的信心與安全網。

--- 第 2 節 ---
在 Python 專案中導入型別註記（Type Hints）能有效提升程式碼的品質與可維護性。透過提前發現錯誤、提供優秀的 IDE 支援、自帶文檔效果以及增強大型專案重構信心，開發者能更穩健地推進專案。在現有專案中，建議採取安裝設定靜態檢查工具 Mypy、從核心模組由內而外開始，以及逐步調高檢查嚴格度並整合 CI/CD 的策略。

--- 第 3 節 ---
在 Python 專案中導入型別註記雖然能大幅提升程式碼品質，但常落入追求百分之百覆蓋率導致開發效率停滯，或是誤將型別註記等同於執行期型別安全保證等陷阱與迷思中。正確的做法是採取漸進式且具實用性的標準，並在需要執行期驗證時搭配 Pydantic 或 Dataclasses 等工具。


## 4. 上線前的檢查清單

這十二章教的是「怎麼做出來」。真的要上線還缺這些，全部在 30 天實作軌：

| 缺什麼 | 去哪學 |
|---|---|
| 成本失控（每輪重送全部歷史） | Day 10 上下文壓縮、Day 11 快取 |
| 沒有護欄、沒有稽核 | Day 12 Plugins、Day 28 企業級安全 |
| 不知道品質有沒有退步 | Day 26 評估、Day 27 模擬器 |
| 出事了查不到原因 | Day 29 Log / Metric / Trace |
| 跑在本機、沒有部署 | Day 30 部署 |
| 要接外部系統 | Day 07 MCP / OpenAPI、Day 19-20 A2A |
| 要人工介入審核 | Day 15 Human-in-the-Loop |

### 一個最低限度的自我檢查

In [7]:
checklist = [
    ("每個 agent 的 description 都寫得能跟兄弟區分嗎？", "第 01、09 章"),
    ("模型有掛 retry_options 嗎？", "第 03 章"),
    ("state 裡有沒有塞大檔案？（該用 Artifact）", "第 05 章"),
    ("護欄是寫成 Plugin 還是 agent callback？", "第 06 章"),
    ("LoopAgent 有設 max_iterations 嗎？", "第 07 章"),
    ("流程固定的部分有沒有誤用 LLM 委派？", "第 08、09 章"),
    ("模型 ID 是集中管理還是散在各處？", "第 00、03 章"),
    ("結構化輸出是從回傳值拿還是從 state 拿？型別不一樣", "第 03、11 章"),
]
for question, where in checklist:
    print(f"  ☐ {question:44s} → {where}")

  ☐ 每個 agent 的 description 都寫得能跟兄弟區分嗎？           → 第 01、09 章
  ☐ 模型有掛 retry_options 嗎？                        → 第 03 章
  ☐ state 裡有沒有塞大檔案？（該用 Artifact）                 → 第 05 章
  ☐ 護欄是寫成 Plugin 還是 agent callback？              → 第 06 章
  ☐ LoopAgent 有設 max_iterations 嗎？               → 第 07 章
  ☐ 流程固定的部分有沒有誤用 LLM 委派？                         → 第 08、09 章
  ☐ 模型 ID 是集中管理還是散在各處？                           → 第 00、03 章
  ☐ 結構化輸出是從回傳值拿還是從 state 拿？型別不一樣                 → 第 03、11 章


In [8]:
# 清掉示範用的專案目錄
shutil.rmtree(DEMO_DIR, ignore_errors=True)
print("已清除", DEMO_DIR.name)

已清除 _capstone_demo


## 本章重點

- **ADK 專案有硬性結構**：`__init__.py` 要 `from . import agent`，
  `agent.py` 裡的變數一定要叫 `root_agent`。
- **`adk run` 可以吃管線輸入**，適合接 CI；`adk web` 是開發期最有用的工具，
  但官方明確標示不可用於正式環境。
- **三層是可以疊起來用的**：Sequential 包 Parallel、最後用 output_schema 收斂。
- **「做得出來」和「上得了線」是兩件事**，後者在 30 天實作軌。

## 動手練習

1. 幫第 1 節的 `faq_agent` 加一個 `_FAQ` 查不到時「轉真人」的分支。
2. 把第 3 節的 `assembler` 換成 `Workflow` 圖，讓「文件太長」時多跑一個
   精簡節點（提示：第 10 章的 `ctx.route`）。
3. 用 `adk web` 打開第 1 節的專案，比較它的事件檢視跟我們的 `trace=True`。

---
## 🎓 概念軌完成

你現在懂了 ADK 的三層結構、兩種多 agent 風格、以及 2.0 的圖形引擎。

**接下來 → [`../30day_practice/`](../30day_practice/)**：
30 天實作軌，一天一個主題，深入每個元件的細節與生產環境的考量。